In [1]:
pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pyspark.sql import SparkSession
from neo4j import GraphDatabase
import json

# 1. Setup Neo4j Cloud Connection
# Replace these with the details from your AuraDB credentials file
CLOUD_URI = "neo4j+s://7bb39fe0.databases.neo4j.io"
CLOUD_USER = "7bb39fe0"
CLOUD_PASS = "5wCN9zh1-kH5f4ZODGGvU4V0i-DyreUtxfBHkjjIjis"

def run_neo4j_loader():
    spark = SparkSession.builder.appName("Neo4jLoader").getOrCreate()
    
    try:
        # Read the cleaned report from your local folder or HDFS
        df = spark.read.parquet("./output/master_report_final")
        data = [row.asDict() for row in df.collect()]
        
        # Connect using the cloud URI and credentials
        with GraphDatabase.driver(CLOUD_URI, auth=(CLOUD_USER, CLOUD_PASS)) as driver:
            with driver.session() as session:
                print("Connected to Neo4j AuraDB. Clearing old graph...")
                session.run("MATCH (n) DETACH DELETE n")
                
                print(f"Uploading {len(data)} batches to the cloud...")
                
                for record in data:
                    session.run("""
                        MERGE (r:Recipe {name: $recipe})
                        MERGE (b:Batch {id: $batch_id, cook_time: $kitchen_mins, loss: $weight_loss_kg})
                        MERGE (d:Driver {id: $driver})
                        MERGE (c:Canteen {id: $canteen})
                        
                        MERGE (r)-[:PRODUCED]->(b)
                        MERGE (b)-[:DELIVERED_BY]->(d)
                        MERGE (d)-[:ARRIVED_AT]->(c)
                    """, 
                    recipe=record['recipe'],
                    batch_id=record['batch_id'],
                    kitchen_mins=record['kitchen_mins'],
                    weight_loss_kg=record['weight_loss_kg'],
                    driver=record['driver'],
                    canteen=record['canteen']
                    )
                
                print("Success! Your cloud graph is ready for analysis.")

    except Exception as e:
        print(f"Cloud Connection Error: {e}")
        print("Tip: Check if your password is correct and your internet is active.")
    finally:
        spark.stop()

if __name__ == "__main__":
    run_neo4j_loader()

26/03/20 16:34:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/20 16:34:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Connected to Neo4j AuraDB. Clearing old graph...
Uploading 55 batches to the cloud...
Success! Your cloud graph is ready for analysis.


In [6]:
from neo4j import GraphDatabase
import pandas as pd

# 1. Connection Details
# Double-check your AuraDB URI from the console to avoid the DNS error
URI = "neo4j+s://7bb39fe0.databases.neo4j.io"
USER = "7bb39fe0"
PASSWORD = "5wCN9zh1-kH5f4ZODGGvU4V0i-DyreUtxfBHkjjIjis"

def run_query(cypher_query, params=None):
    """Helper function to run a query and return a Pandas DataFrame"""
    with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
        # result_as_df is a shortcut to get clean tables in Jupyter
        records, summary, keys = driver.execute_query(
            cypher_query,
            database_="7bb39fe0",
            parameters_=params
        )
        return pd.DataFrame([r.data() for r in records])

# --- Query 1: Busiest Kitchen Stations (By Recipe) ---
print("--- 1. Busiest Kitchen Stations ---")
q1 = """
MATCH (r:Recipe)-[:PRODUCED]->(b:Batch)
RETURN r.name AS Recipe, count(b) AS Total_Batches
ORDER BY Total_Batches DESC
"""
display(run_query(q1))

# --- Query 2: High-Risk Drivers (High Weight Loss Deliveries) ---
# This finds drivers handling batches with more than 0.5kg weight loss
print("\n--- 2. Drivers Handling High Weight-Loss Batches ---")
q2 = """
MATCH (b:Batch)-[:DELIVERED_BY]->(d:Driver)
WHERE b.loss > 0.5
RETURN d.id AS Driver_ID, count(b) AS Risky_Deliveries, avg(b.loss) AS Avg_Loss
ORDER BY Risky_Deliveries DESC
"""
display(run_query(q2))

# --- Query 3: Canteen Delivery Volume ---
print("\n--- 3. Canteen Traffic Report ---")
q3 = """
MATCH (d:Driver)-[:ARRIVED_AT]->(c:Canteen)
RETURN c.id AS Canteen, count(d) AS Total_Deliveries
ORDER BY Total_Deliveries DESC
"""
display(run_query(q3))

# --- Query 4: Path Trace (Full Journey of a Batch) ---
print("\n--- 4. Journey Trace for BATCH-1001 ---")
q4 = """
MATCH path = (r:Recipe)-[:PRODUCED]->(b:Batch {id: 'BATCH-1001'})-[:DELIVERED_BY]->(d:Driver)-[:ARRIVED_AT]->(c:Canteen)
RETURN r.name AS Recipe, b.id AS Batch, d.id AS Driver, c.id AS Canteen
"""
display(run_query(q4))

--- 1. Busiest Kitchen Stations ---


,Recipe,Total_Batches
0,beef_stew,21
1,fish_curry,12
2,chicken_rice,11
3,veg_pasta,11



--- 2. Drivers Handling High Weight-Loss Batches ---


,Driver_ID,Risky_Deliveries,Avg_Loss
0,drv_01,6,1.178333
1,drv_03,5,0.832000
2,drv_04,5,2.318000
3,drv_06,5,1.512000
4,drv_02,4,0.702500
5,drv_05,4,1.360000
6,drv_07,2,0.670000



--- 3. Canteen Traffic Report ---


,Canteen,Total_Deliveries
0,cant_04,7
1,cant_03,6
2,cant_02,6
3,cant_01,5



--- 4. Journey Trace for BATCH-1001 ---


,Recipe,Batch,Driver,Canteen
0,fish_curry,BATCH-1001,drv_02,cant_01
1,fish_curry,BATCH-1001,drv_02,cant_04
2,fish_curry,BATCH-1001,drv_02,cant_03
